# Supervised Fine-Tuning with PubMed Dataset on Microsoft Foundry

This notebook demonstrates how to fine-tune language models using **Supervised Fine-Tuning (SFT)** with the PubMed Medical Research Summarization dataset.

## What You'll Learn
1. Understand supervised fine-tuning for medical text summarization
2. Prepare and format medical research data
3. Upload datasets to Microsoft Foundry
4. Create and monitor a supervised fine-tuning job
5. Deploy and test your fine-tuned model

**Note**: Execute each cell in sequence.

## 1. Setup and Installation

Install all required packages from requirements.txt

In [1]:
pip install -r requirements.txt

Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/
Note: you may need to restart the kernel to use updated packages.


## 2. Import Libraries

In [2]:
import os
import time
from itertools import islice
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

print("All libraries imported successfully")


All libraries imported successfully


## 3. Configure the Microsoft Foundry Project

Set the Microsoft Foundry project endpoint and base model name. Copy `.env.template` to `.env` and provide the values required for your run.

```text
MICROSOFT_FOUNDRY_PROJECT_ENDPOINT=https://<account>.services.ai.azure.com/api/projects/<project>
MODEL_NAME=gpt-4.1

# Optional: reuse an existing terminal job and project deployment.
FINE_TUNE_JOB_ID=<existing-ftjob-id>
DEPLOYMENT_NAME=<existing-project-deployment-name>
```

Authentication uses `DefaultAzureCredential`; no API key, account endpoint, subscription ID, resource group, or account name is needed.


In [3]:
load_dotenv()

project_endpoint = os.environ.get("MICROSOFT_FOUNDRY_PROJECT_ENDPOINT")
model_name = os.environ.get("MODEL_NAME")
if not project_endpoint or not model_name:
    raise RuntimeError("MICROSOFT_FOUNDRY_PROJECT_ENDPOINT and MODEL_NAME must be set")
if "/api/projects/" not in project_endpoint:
    raise ValueError("MICROSOFT_FOUNDRY_PROJECT_ENDPOINT must be a Microsoft Foundry project endpoint")

# Create bounded, disjoint subsets so the demonstration run completes promptly.
def write_subset(source_path, destination_path, count):
    with source_path.open("r", encoding="utf-8") as source, destination_path.open("w", encoding="utf-8") as destination:
        destination.writelines(islice(source, count))

training_file_path = Path("training_demo.jsonl")
validation_file_path = Path("validation_demo.jsonl")
write_subset(Path("training.jsonl"), training_file_path, 10)
write_subset(Path("validation.jsonl"), validation_file_path, 10)
print("Prepared 10-example training and validation subsets for this demonstration run")


Prepared 10-example training and validation subsets for this demonstration run


## 4. Connect to the Microsoft Foundry Project

Create one `AIProjectClient` from the project endpoint and `DefaultAzureCredential`. The authenticated OpenAI-compatible client returned by the project client is used for files, fine-tuning, and inference.

**Important**: Ensure you have the **Azure AI User** role assigned on the Microsoft Foundry project.


In [4]:
credential = DefaultAzureCredential(process_timeout=120)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("Connected to Microsoft Foundry Project")


Connected to Microsoft Foundry Project


## 5. Upload Training Files

Upload the training and validation JSONL files to Microsoft Foundry. Each file is assigned a unique ID that will be referenced when creating the fine-tuning job.

In [5]:
print("Uploading training file...")
with open(training_file_path, "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Training file ID: {train_file.id}")

print("\nUploading validation file...")
with open(validation_file_path, "rb") as f:
    validation_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Validation file ID: {validation_file.id}")

Uploading training file...


Training file ID: file-5b7f953d4a22478d85e28016ce19a3f1

Uploading validation file...


Validation file ID: file-e942cb4ce5c54a6f888eced2230c834e


## 6. Wait for File Processing

Microsoft Foundry needs to process the uploaded files before they can be used for fine-tuning.

In [6]:
print("Waiting for files to be processed...")
openai_client.files.wait_for_processing(train_file.id)
openai_client.files.wait_for_processing(validation_file.id)
print("Files ready!")

Waiting for files to be processed...


Files ready!


## 7. Create Supervised Fine-Tuning Job

Create a supervised fine-tuning job with your uploaded datasets. Configure the following hyperparameters to control the training process:

**Hyperparameters:**
1. **n_epochs (3)**: Number of complete passes through the training dataset. More epochs can improve performance but may lead to overfitting. Typical range: 1-10.
2. **batch_size (1)**: Number of training examples processed together in each iteration. Smaller batches provide more frequent updates. Typical range: 1-8.
3. **learning_rate_multiplier (1.0)**: Scales the default learning rate. Values < 1.0 make training more conservative, while values > 1.0 speed up learning but may cause instability. Typical range: 0.1-2.0.

**Note**: Adjust these based on your dataset size and quality.

In [7]:
existing_job_id = os.environ.get("FINE_TUNE_JOB_ID")
if existing_job_id:
    print(f"Reusing completed fine-tuning job: {existing_job_id}")
    fine_tune_job = openai_client.fine_tuning.jobs.retrieve(existing_job_id)
else:
    print("Creating supervised fine-tuning job...")
    fine_tune_job = openai_client.fine_tuning.jobs.create(
        model=model_name,
        training_file=train_file.id,
        validation_file=validation_file.id,
        method={
            "type": "supervised",
            "supervised": {"hyperparameters": {"n_epochs": 1, "batch_size": 1, "learning_rate_multiplier": 1.0}},
        },
        extra_body={"trainingType": "GlobalStandard"},
        suffix="pubmed-summarization"
    )

print("Fine-tuning job ready!")
print(f"Job ID: {fine_tune_job.id}")
print(f"Status: {fine_tune_job.status}")
print(f"Model: {fine_tune_job.model}")

Reusing completed fine-tuning job: ftjob-72ede626606d40eebb7aaa15060a24ee


Fine-tuning job ready!
Job ID: ftjob-72ede626606d40eebb7aaa15060a24ee
Status: succeeded
Model: gpt-4.1-2025-04-14


## 8. Monitor Training to a Terminal State

Poll the project-scoped fine-tuning API until the job succeeds, fails, or is cancelled. Recent events are printed when the job reaches a terminal state.


In [8]:
terminal_statuses = {"succeeded", "failed", "cancelled"}
while True:
    job_status = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id)
    print(f"Status: {job_status.status}")
    if job_status.status in terminal_statuses:
        break
    time.sleep(60)

for event in reversed(list(openai_client.fine_tuning.jobs.list_events(fine_tune_job.id, limit=10))):
    print(f"{event.created_at}: {event.level} - {event.message}")

if job_status.status != "succeeded":
    raise RuntimeError(f"Fine-tuning job {fine_tune_job.id} ended with status {job_status.status}: {job_status.error}")


Status: succeeded


1790988348: info - Job enqueued. Waiting for jobs ahead to complete.
1790988480: info - Preprocessing running for file training file.
1790988880: info - Preprocessing running for file validation file.
1790988881: info - Preprocessing completed for file training file.
1790988881: info - Preprocessing completed for file validation file.
1790988897: info - Job started.
1790988940: info - Data Import started.
1790988940: info - Finetuning started.
1790989527: info - Job queued, waiting for GPUs.
1790989763: info - Training started.
1790990711: info - Step 1: training loss=1.4209126234054565
1790990772: info - Created results file: file-4df718eccc06470186009dc170cd3cc8
1790991198: info - Step 10: training loss=0.8785944581031799
1790993133: info - Job succeeded.
1790993165: info - Completed results file: file-4df718eccc06470186009dc170cd3cc8
1790993165: info - Model Evaluation Passed.
1790993165: info - Training tokens billed: 46000


## 9. Retrieve Fine-Tuned Model

After the fine-tuning job succeeded, retrieve the fine-tuned model ID. This ID is required to make inference calls with your customized model.

In [9]:
completed_job = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id)

if completed_job.status == "succeeded":
    fine_tuned_model_id = completed_job.fine_tuned_model
    print(f"Fine-tuned Model ID: {fine_tuned_model_id}")
else:
    raise RuntimeError(f"Fine-tuning job ended with status {completed_job.status}: {completed_job.error}")

Fine-tuned Model ID: gpt-4.1-2025-04-14.ft-72ede626606d40eebb7aaa15060a24ee-pubmed-summarization

## 10. Resolve and Monitor the Project Deployment

`azure-ai-projects` 2.x enumerates deployments available to the Foundry project. Resolve the deployment that serves the completed fine-tuned model, optionally using `DEPLOYMENT_NAME`, and poll until it is visible through the project endpoint.


In [10]:
configured_deployment_name = os.environ.get("DEPLOYMENT_NAME")
deployment = None
deployment_deadline = time.monotonic() + 600

while time.monotonic() < deployment_deadline:
    deployments = list(project_client.deployments.list())
    if configured_deployment_name:
        deployment = next((item for item in deployments if item.name == configured_deployment_name), None)
    else:
        deployment = next(
            (item for item in deployments if getattr(item, "model_name", None) == fine_tuned_model_id),
            None,
        )
    if deployment:
        break
    print("Waiting for the fine-tuned model deployment to appear in the project...")
    time.sleep(30)

if deployment is None:
    raise RuntimeError(
        "No project deployment was found for the fine-tuned model. Create it in the same Foundry project "
        "and optionally set DEPLOYMENT_NAME before rerunning this section."
    )

deployment_name = deployment.name
resolved_deployment = project_client.deployments.get(deployment_name)
if getattr(resolved_deployment, "model_name", None) != fine_tuned_model_id:
    raise RuntimeError(
        f"Deployment {deployment_name} serves {getattr(resolved_deployment, 'model_name', None)}, "
        f"not {fine_tuned_model_id}"
    )

print(f"Deployment ready in project: {deployment_name}")
print(f"Deployed model: {resolved_deployment.model_name}")


Deployment ready in project: pubmed-060a24ee
Deployed model: gpt-4.1-2025-04-14.ft-72ede626606d40eebb7aaa15060a24ee-pubmed-summarization


## 11. Monitor Inference Readiness and Test the Fine-Tuned Model

Use the same project-scoped client to poll inference until the deployment responds, then print a PubMed-style summary.


In [11]:
test_article = """Background: Alzheimer's disease (AD) is a progressive neurodegenerative disorder characterized by cognitive decline and memory loss. Recent studies have suggested that neuroinflammation plays a crucial role in disease pathogenesis. Objective: This study investigated the role of microglial activation and inflammatory cytokines in early-stage Alzheimer's disease progression. Methods: We conducted a longitudinal study of 150 patients with mild cognitive impairment (MCI) over 36 months. Cerebrospinal fluid (CSF) samples were analyzed for inflammatory markers including IL-6, TNF-alpha, and IL-1beta. Brain imaging using PET scans assessed microglial activation. Results: Patients who progressed to AD showed significantly elevated levels of IL-6 (p<0.001) and TNF-alpha (p<0.01) at baseline compared to stable MCI patients. Microglial activation was observed in the hippocampus and entorhinal cortex regions. Multivariate analysis revealed that combined inflammatory markers predicted AD conversion with 78% accuracy. Conclusion: Elevated neuroinflammatory markers and microglial activation in MCI patients are associated with increased risk of progression to Alzheimer's disease, suggesting potential therapeutic targets for early intervention."""

inference_deadline = time.monotonic() + 1200
last_inference_error = None
while time.monotonic() < inference_deadline:
    try:
        response = openai_client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a medical research summarization assistant."},
                {"role": "user", "content": f"Summarize this medical research article:\n\n{test_article}"},
            ],
            temperature=0.7,
            max_completion_tokens=500,
        )
        break
    except Exception as exc:
        last_inference_error = exc
        print(f"Deployment not inference-ready yet: {type(exc).__name__}")
        time.sleep(30)
else:
    raise TimeoutError(f"Deployment {deployment_name} did not become inference-ready: {last_inference_error}")

summary = response.choices[0].message.content
if not summary:
    raise RuntimeError("Inference completed without summary content")
print(f"Inference succeeded with deployment: {deployment_name}")
print(summary)


Inference succeeded with deployment: pubmed-060a24ee
**Summary:**  
This longitudinal study followed 150 patients with mild cognitive impairment (MCI) for 36 months to investigate the role of neuroinflammation in early Alzheimer's disease (AD) progression. Researchers measured inflammatory cytokines (IL-6, TNF-alpha, IL-1beta) in cerebrospinal fluid and used PET imaging to assess microglial activation. Patients who progressed from MCI to AD had significantly higher baseline levels of IL-6 and TNF-alpha, and showed microglial activation in the hippocampus and entorhinal cortex. Combined inflammatory markers predicted AD conversion with 78% accuracy. The findings indicate that elevated neuroinflammatory markers and microglial activation are associated with increased risk of progression to AD, highlighting potential therapeutic targets for early intervention.


##  Congratulations!

You've successfully fine-tuned a model for medical research summarization using the PubMed dataset!

### Next Steps:
1. **Test with more examples**: Try different medical research articles to evaluate performance
2. **Adjust hyperparameters**: Experiment with different epoch counts, batch sizes, or learning rates
3. **Deploy to production**: Integrate your fine-tuned model into healthcare applications
4. **Fine-tune further**: Use your own medical publications or clinical notes for specialized summarization
5. **Explore use cases**: Apply the model to systematic reviews, literature analysis, or clinical decision support